# генерация синтетических данных для датасета

полное обучение классификатора с интеграцией CleaML и задачей генерации синтетических данных с помощью LLM Qwen3 на моем гитхабе: https://github.com/poeeeri/Banking-Support

# обучение BERT классификатора

In [35]:
!pip install -q transformers datasets accelerate

In [36]:
import pandas as pd
import numpy as np
import common
from pathlib import Path
import json
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [37]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

## препроцессинг текста

функция, берущая последнее сообщение пользователя

In [38]:
def extract_last_user_message(dialog):
    lines = [line.strip() for line in dialog.splitlines() if line.strip()]
    user_lines = [line for line in lines if line.lower().startswith("user:")]
    if not user_lines:
        return ""
    return user_lines[-1].split(":", 1)[1].strip()

преобразует строку в список, где каждый элемент - это сообщение

In [39]:
def dialog_turns(dialog):
    turns: list[tuple[str, str]] = []
    for raw_line in dialog.splitlines():
        line = raw_line.strip()
        if not line or ":" not in line:
            continue
        speaker, text = line.split(":", 1)
        speaker = speaker.strip().lower()
        text = text.strip()
        if speaker in {"user", "assistant"} and text:
            turns.append((speaker, text))
    return turns


In [40]:
def load_jsonl(path):
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            text = line.strip()
            if text:
                rows.append(json.loads(text))
    return rows

In [41]:
def build_last_message(dialog):
    return extract_last_user_message(dialog)

In [42]:
def normalize_dialog(dialog: str):
    turns = dialog_turns(dialog)
    return "\n".join(f"{speaker}: {text}" for speaker, text in turns)

даем берт трансформеру контекст о всем диалоге поддержки с пользователем, но делаем акцент на последнее сообщение, чтобы дать понять модели, его значимость

In [43]:
def build_training_text(dialog):
    normalized = normalize_dialog(dialog)
    last_user_message = extract_last_user_message(normalized)
    return (
        "Ниже диалог клиента с банковской поддержкой.\n"
        "Определи интент по последнему сообщению пользователя с учетом контекста.\n\n"
        f"{normalized}\n\n"
        f"Последнее сообщение пользователя: {last_user_message}"
    )

In [44]:
def prepare_rows(records):
    prepared: list[dict] = []
    for row in records:
        text = build_training_text(row["dialog"])
        prepared.append(
            {
                "text": text,
                "labels": LABEL_TO_ID[row["label"]],
                "label": row["label"],
                "dialog": row["dialog"],
                "last_user_message": build_last_message(row["dialog"]),
            }
        )
    return prepared

In [45]:
ID_TO_LABEL = common.ID_TO_LABEL
LABEL_TO_ID = common.LABEL_TO_ID
INTENT_SPECS = common.INTENT_SPECS

SEED = 42
MODEL_NAME = 'DeepPavlov/rubert-base-cased'

In [46]:
set_seed(SEED)

## обучение

преобразуем пайтон списки в хаггинг фейс датасет, чтобы подать их в трейнер

In [47]:
train_records = prepare_rows(load_jsonl(Path('/content/data/train.jsonl')))
test_records = prepare_rows(load_jsonl(Path('/content/data/test.jsonl')))

train_dataset = Dataset.from_list(train_records)
test_dataset = Dataset.from_list(test_records)

In [48]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(INTENT_SPECS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

в этой функции берем каждый батч и переводим их в токены через инициализированный токенизатор

In [49]:
def tokenize_batch(batch, tokenizer, max_length):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
    )
    tokenized["labels"] = batch["labels"]
    return tokenized

берем как максимум 256 токенов, так как диалоги предполагаются короткие

In [50]:
tokenized_train = train_dataset.map(lambda batch: tokenize_batch(batch, tokenizer, 256), batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(lambda batch: tokenize_batch(batch, tokenizer, 256), batched=True, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [51]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }

In [52]:
def sanitize_metrics(metrics):
    sanitized = {}
    for key, value in metrics.items():
        if isinstance(value, (np.floating, np.integer)):
            sanitized[key] = value.item()
        else:
            sanitized[key] = value
    return sanitized

In [53]:
training_args = TrainingArguments(
    output_dir = '/content/artifacts',
    learning_rate = 2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5.0,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True, # задаем правило чем больше nacro f1, тем лучше
    report_to="none",
    save_total_limit=2
)
# сид установила в начале ноутбука

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [54]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
eval_netrics = trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.699681,2.612952,0.160000,0.059991,0.059991
2,2.397785,2.475621,0.153333,0.111811,0.111811
3,2.030637,1.934481,0.746667,0.749129,0.749129
4,1.656205,1.539489,0.926667,0.927019,0.927019
5,1.423429,1.359599,0.960000,0.960143,0.960143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

In [58]:
trainer.save_model(Path('/content/artifacts/best_model'))
tokenizer.save_pretrained(Path('/content/artifacts/best_model'))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/artifacts/best_model/tokenizer_config.json',
 '/content/artifacts/best_model/tokenizer.json')

In [59]:
prediction_output = trainer.predict(tokenized_test)
predictions = np.argmax(prediction_output.predictions, axis=-1)
references = np.array(test_dataset["labels"])

In [63]:
report = classification_report(
    references,
    predictions,
    labels=list(range(len(INTENT_SPECS))),
    target_names=['transfer_status', 'transfer_cancel', 'card_block', 'card_unblock',
                  'card_delivery', 'credit_card_making', 'password_reset', 'changing_account_information',
                  'credit_card_limit_changing', 'account_statement', 'bill_repayment',
                  'loan_early_repayment', 'loan_application_status', 'new_loan_application',
                  'exchange_rate'],
    output_dict=True,
    zero_division=0,
)

In [65]:
prediction_rows = []
for row, pred_id in zip(test_records, predictions):
  prediction_rows.append(
      {
          "gold_label": row["label"],
          "predicted_label": ID_TO_LABEL[int(pred_id)],
          "last_user_message": row["last_user_message"],
          "dialog": row["dialog"]
      }
  )

metrics_payload = {
    "train_runtime": train_result.metrics,
    "eval_metrics": sanitize_metrics(eval_netrics),
    "classification_report": report
}

In [66]:
metrics_payload

{'train_runtime': {'train_runtime': 612.7035,
  'train_samples_per_second': 2.448,
  'train_steps_per_second': 0.31,
  'total_flos': 171691779228768.0,
  'train_loss': 2.141803731416401,
  'epoch': 5.0},
 'eval_metrics': {'eval_loss': 1.3595993518829346,
  'eval_accuracy': 0.96,
  'eval_macro_f1': 0.9601427811954129,
  'eval_weighted_f1': 0.9601427811954129,
  'eval_runtime': 2.1559,
  'eval_samples_per_second': 69.576,
  'eval_steps_per_second': 4.638,
  'epoch': 5.0},
 'classification_report': {'transfer_status': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 10.0},
  'transfer_cancel': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 10.0},
  'card_block': {'precision': 0.7777777777777778,
   'recall': 0.7,
   'f1-score': 0.7368421052631579,
   'support': 10.0},
  'card_unblock': {'precision': 0.75,
   'recall': 0.9,
   'f1-score': 0.8181818181818182,
   'support': 10.0},
  'card_delivery': {'precision': 1.0,
   'recall': 1.0,
   'f1-sco

In [67]:
!zip -r /content/artifacts/best_model.zip /content/artifacts/best_model


  adding: content/artifacts/best_model/ (stored 0%)
  adding: content/artifacts/best_model/model.safetensors (deflated 7%)
  adding: content/artifacts/best_model/tokenizer.json (deflated 73%)
  adding: content/artifacts/best_model/config.json (deflated 63%)
  adding: content/artifacts/best_model/training_args.bin (deflated 53%)
  adding: content/artifacts/best_model/tokenizer_config.json (deflated 42%)


In [68]:
from google.colab import files
files.download("/content/artifacts/best_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
i = 109

row = test_records[i]
pred_id = predictions[i]
ref_id = references[i]


print("=" * 100)
print("ref:", ID_TO_LABEL[int(ref_id)])
print("pred:", ID_TO_LABEL[int(pred_id)])
print(row["dialog"])

ref: bill_repayment
pred: bill_repayment
user: У меня есть счет на оплату, как я могу его оплатить?
assistant: Конечно, можно оплатить через интернет-банк, мобильное приложение или в любом отделении банка. Если вы используете онлайн-сервис, платеж обычно зачисляется в течение 1-2 рабочих дней.
user: А если платеж не прошёл, что делать?


In [90]:
test_metrics = prediction_output.metrics
test_metrics

{'test_loss': 1.3595993518829346,
 'test_accuracy': 0.96,
 'test_macro_f1': 0.9601427811954129,
 'test_weighted_f1': 0.9601427811954129,
 'test_runtime': 2.2919,
 'test_samples_per_second': 65.449,
 'test_steps_per_second': 4.363}

проверим пример, которого нет нигде из датасетов

In [89]:
import torch

device = next(model.parameters()).device
print(device)

my_dialog = """user: Добрый день, я подавала заявку на кредит.
assistant: Здравствуйте! Когда вы отправили заявку?
user: Заявку отправила вчера вечером, но ответа пока нет. Можете проверить, на каком она этапе?"""

text = build_training_text(my_dialog)

encoded = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

encoded = {key: value.to(device) for key, value in encoded.items()}

model.eval()
with torch.no_grad():
    logits = model(**encoded).logits
    probs = torch.softmax(logits, dim=-1)[0]

pred_id = int(probs.argmax().item())

print("pred:", ID_TO_LABEL[pred_id])
print("prob:", float(probs[pred_id].detach().cpu()))

cuda:0
pred: loan_application_status
prob: 0.1983620971441269
